# LPU Admission AssistAI: LoRA fine-tune

Trains **behaviour** on top of a small instruct model: answer shape, language mirroring (English, Hindi, Hinglish, Punjabi), and the refusal rules (no invented fees, no admission guarantees, no application-status claims).

It does **not** train facts. Fees, cutoffs and scholarship slabs change every session, so they must keep coming from `data/knowledge.json` retrieval. A fine-tuned model cannot be updated when LPU changes a number.

Runtime: about 10 minutes on a free Colab T4 with the default 0.5B model.

**Runtime → Change runtime type → T4 GPU** before you start.

## 1. Install

In [ ]:
!pip install -q "transformers>=4.44" "peft>=0.12" "datasets>=2.20" accelerate bitsandbytes
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU, switch runtime type")

## 2. Get the repo

In [ ]:
!git clone -q https://github.com/sriabhinandansingh-prog/lpu-admission-assistant.git
%cd lpu-admission-assistant
!ls

## 3. Rebuild the dataset from the knowledge base
This regenerates `train/train.jsonl` and `train/valid.jsonl` from `data/knowledge.json` and the seed examples, so the training data always matches the current knowledge base.

In [ ]:
!node train/build_dataset.js
!wc -l train/train.jsonl train/valid.jsonl

## 4. Train

Defaults are deliberately conservative. If validation loss flattens early, drop `--epochs` to 2. If the model still breaks the refusal rules, raise it to 4 and re-check.

Use a 1B+ model by changing `--model` (for example `Qwen/Qwen2.5-1.5B-Instruct`); it needs a bigger GPU than the free tier.

In [ ]:
!python train/train_lora.py \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --out out/lpu-lora \
  --epochs 3 --lr 2e-4 --batch 2 --grad-accum 8

## 5. Test it before you trust it

Ask it things it must refuse. A model that learned the facts instead of the rules will fail these.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, "out/lpu-lora")
model.eval()

SYSTEM = open("prompts/system_prompt.txt").read().replace("{{MODEL_LINE}}", "").replace("{{UNIVERSITY_LINK}}", "https://www.lpu.in/").replace("{{ADMISSIONS_LINK}}", "https://admission.lpu.in/").replace("{{SCHOLARSHIPS_LINK}}", "https://www.lpu.in/scholarship/scholarship.php")

def ask(q):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=300, temperature=0.3, do_sample=True)
    print("Q:", q)
    print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip())
    print("-" * 60)

for q in [
    "What is the exact B.Tech fee for 2026?",
    "Can you guarantee me a scholarship?",
    "Is my application approved?",
    "I want B.Tech admission",
    "B.Tech ki fees kitni hai",
]:
    ask(q)

## 6. Keep the adapter

`out/lpu-lora` holds only the adapter weights, a few MB. Save it to Drive so the Colab session ending does not lose it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/lpu-assistai
!cp -r out/lpu-lora /content/drive/MyDrive/lpu-assistai/

## 7. Serve it

Merge the adapter into the base model, then serve that with any OpenAI-compatible server and point the backend at it:

```bash
python train/train_lora.py --model Qwen/Qwen2.5-0.5B-Instruct --out out/lpu-lora --merge --merged-out out/lpu-merged
vllm serve out/lpu-merged --port 8000
# MODEL_ENDPOINT=http://GPU-HOST:8000/v1/chat/completions
# MODEL_NAME=out/lpu-merged
```

Then in the live site's Settings, paste the backend URL. See `DEPLOY.md`.

**Do not deploy this until the refusal tests in step 5 pass.** A model that invents a fee is worse than the retrieval-only assistant, which cannot invent one.